In [1]:
pip install osmnx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 8.7 MB/s eta 0:00:00


In [2]:
import osmnx as ox
import networkx as nx
import folium
import random

In [21]:
place = "Quận 10, Hồ Chí Minh, Việt Nam"
G = ox.graph_from_place(place, network_type="drive")
origin_node = random.choice(list(G.nodes()))
destination_node = random.choice(list(G.nodes()))
route1 = nx.shortest_path(G, origin_node, destination_node, weight='length')
congestion_area = route1[len(route1)//2]

In [22]:
for u, v, key, data in G.edges(keys=True, data=True):
  origin_cost = data['length']
  risk = 1
  try:
    if nx.shortest_path_length(G, congestion_area, u) <=5:
      risk = 10
  except nx.NetworkXNoPath:
    pass
  data['risk_cost'] = origin_cost * risk
route2 = nx.shortest_path(G, origin_node, destination_node, weight='risk_cost')

In [25]:
m = folium.Map(location=(G.nodes[origin_node]['y'], G.nodes[origin_node]['x']), zoom_start=13)

folium.Circle(
    location=[G.nodes[congestion_area]['y'], G.nodes[congestion_area]['x']],
    radius=400,
    color='red',
    fill_color=True,
    fill_opacity=0.3,
    tooltip='Vùng có nguy cơ tắc nghẽn giao thôn'
).add_to(m)

folium.PolyLine(
    locations=[(G.nodes[n]['y'], G.nodes[n]['x']) for n in route1],
    color='orange',
    tooltip='Tuyến đường có nguy cơ kẹt xe'
).add_to(m)

folium.PolyLine(
    locations=[(G.nodes[n]['y'], G.nodes[n]['x']) for n in route2],
    color='green',
    tooltip='Tuyến đường đề xuất'
).add_to(m)

folium.Marker((G.nodes[origin_node]['y'], G.nodes[origin_node]['x']), tooltip='Điểm xuất phát', icon=folium.Icon(color='blue', icon='home')).add_to(m)
folium.Marker((G.nodes[destination_node]['y'], G.nodes[destination_node]['x']), tooltip='Đích đến', icon=folium.Icon(color='red', icon='flag')).add_to(m)

In [26]:
m